# Equivalence check: merged writer vs original run

Drafts 200 seeded validation reports with the merged LoRA r16 writer, the way the demo does,
and compares them with the stored predictions of the original run. Run on a **GPU** runtime
(Runtime > Change runtime type > A100). The result file has counts only, no report text.

In [ ]:
# 1. Your backup archive on Google Drive (rrs_runs_2026-11-25.tar.gz). Edit the path if needed.
from google.colab import drive
drive.mount('/content/drive')
BACKUP = '/content/drive/MyDrive/rrs_runs_2026-11-25.tar.gz'
!tar -xzf "$BACKUP" -C /content --wildcards 'runs_from_rangpur/lora_r16_lr3e-3/*'
!ls -la /content/runs_from_rangpur/lora_r16_lr3e-3 /content/runs_from_rangpur/lora_r16_lr3e-3/best

In [ ]:
# 2. Code and pinned libraries (Colab's own torch is kept; its version is recorded in the result).
!git clone -q https://github.com/BrianTr9/Radiology-Lay-Summariser.git /content/repo
!pip install -q transformers==5.19.0 tokenizers==0.23.3 huggingface_hub==2.2.0 peft==0.21.2 sentencepiece==0.2.2 datasets==5.1.0 rouge_score==0.1.2
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

In [ ]:
# 3. Run (about 10 minutes on an A100). If pip asked for a restart, restart the runtime and run cells 1 to 3 again.
%cd /content/repo
!python eval/equivalence.py \
    --adapter /content/runs_from_rangpur/lora_r16_lr3e-3/best \
    --predictions /content/runs_from_rangpur/lora_r16_lr3e-3/val_predictions.jsonl

In [ ]:
# 4. Show the result and download it; send eval/results/equivalence.json back.
import json
print(json.dumps(json.load(open('/content/repo/eval/results/equivalence.json')), indent=1)[:3000])
from google.colab import files
files.download('/content/repo/eval/results/equivalence.json')